# B01 — Train ResNet-50, T00
**2A202602810 · Nguyễn Thái Anh**

Train từ ImageNet pretrained đủ **12 epoch**, seed 0, batch 64, ảnh 224, AMP.
Giữ fold 0; chọn checkpoint bằng macro-F1 val cao nhất, hòa lấy epoch sớm. Test tắt.

**Đợt 3: Chạy cuối, sau khi hai cặp trước hoàn tất.**
Peak reserved khi chạy riêng trong pilot: **4.20 GiB**.
Mỗi notebook dùng một kernel riêng, cùng interpreter .venv; train chạy trong process con riêng để tách RNG và giải phóng GPU khi xong.

Mở hai notebook cùng đợt và Run All ở cả hai, tối đa hai job trên RTX 3060.
Chờ **cả hai** hoàn tất rồi bắt đầu đợt sau. Nếu hai notebook nối cùng một kernel đang bận,
chọn kernel/session độc lập; tên interpreter giống nhau vẫn được.

Notebook tự resume từ last.pt với cùng config. Khi đã hoàn thành, Run All chỉ đọc và kiểm tra kết quả.
Output theo ID riêng; không dùng checkpoint pilot hoặc AMP audit để khởi tạo run B này.

## 1. Môi trường
Chọn DeepWeeds (.venv) hoặc interpreter .venv/bin/python trên host có CUDA. ROOT_HINT chỉ cần khi không tìm được gốc repo từ thư mục đang mở.

In [ ]:
import os, sys, json
from pathlib import Path
from dataclasses import asdict
ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo có eval.py và starter/")
SUBMISSION = ROOT / "submissions/2A202602810_NguyenThaiAnh"
CODE = SUBMISSION / "code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
import pandas as pd
import numpy as np
from IPython.display import display
import backbone_notebook as nbtrain
import train as trainer
assert Path(trainer.__file__).resolve() == (CODE / "train.py").resolve()
print("ROOT:", ROOT)
print("Python:", sys.executable)

## 2. Cấu hình dùng chung T00
RUN_TRAINING=True sẽ chạy đủ epoch khi Run All; False chỉ xem trạng thái/kết quả đã có.
Giữ seed/epoch/batch/LR/augmentation/loss giống nhau giữa cả 5 notebook.
Nếu muốn cấu hình khác, thay đồng đều ở 5 notebook và dùng EXP_ID mới.
Không giảm batch giữa chừng khi resume: config và scheduler cần giữ nguyên.

In [ ]:
EXP_ID = "B01"
BACKBONE = "resnet50"
PLANNED_GROUP = 3
SEED = 0
EPOCHS = 12
BATCH_SIZE = 64
NUM_WORKERS = 2
RUN_TRAINING = True

cfg = nbtrain.t00_config(EXP_ID, BACKBONE, seed=SEED, epochs=EPOCHS,
                         batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
trainer.validate_config(cfg)
display(pd.DataFrame([asdict(cfg)]).T.rename(columns={0:"value"}))
print("Run:", trainer.run_dir(cfg))
print("Prediction val:", trainer.pred_path(cfg, "val"))

## 3. Kiểm tra điều kiện đầu vào
Sanity và pilot 6.2 đã đạt; AMP audit xác nhận 5 model ổn định.
Train sẽ kiểm tra lại original CSV hashes và split trước khi bắt đầu.
Bảng lịch dưới đây dùng peak reserved của các run riêng lẻ, chưa gồm CUDA contexts hoặc đỉnh khi chạy cùng nhau.

In [ ]:
sanity_path = CODE / "validation/sanity/summary.json"
pilot_folder = CODE / "validation/pilot62/P62_session01"
amp_path = CODE / "validation/amp_audit/P62_session01/summary.json"
sanity = json.loads(sanity_path.read_text())
pilots = json.loads((pilot_folder / "pilot_results.json").read_text())
amp_audit = json.loads(amp_path.read_text())
assert sanity["all_passed"]
assert len(pilots) == 5 and all(r["status"] == "completed" and r["outputs_verified"] for r in pilots)
assert amp_audit["all_passed"] and amp_audit["source_session"] == "P62_session01"
assert {r["source_exp_id"] for r in amp_audit["results"]} == {r["exp_id"] for r in pilots}
original_hashes = json.loads((pilot_folder / "split_hashes.json").read_text())
import hashlib
current_hashes = {name: hashlib.sha256((Path(cfg.labels_dir) / name).read_bytes()).hexdigest()
                  for name in original_hashes}
assert current_hashes == original_hashes, "CSV thay đổi so với pilot"
trainer._verify_provenance(current_hashes, cfg.fold)
by_model = {r["backbone"]:r for r in pilots}
schedule = [
    (1, "B03 ConvNeXt-Tiny + B04 DeiT-Small", ["convnext_tiny", "deit_small_patch16_224"]),
    (2, "B02 ResNeXt-50 + B05 EfficientNet-B0", ["resnext50_32x4d", "efficientnet_b0"]),
    (3, "B01 ResNet-50", ["resnet50"]),
]
display(pd.DataFrame([{"đợt":number, "models":names,
                       "sum_pilot_reserved_GiB":sum(by_model[m]["peak_reserved_gib"] for m in members)}
                      for number,names,members in schedule]).round(2))
print("Preflight: PASS; CSV nguyên bản, sanity/pilot/AMP audit đạt.")

## 4. Trạng thái run và GPU
New: train từ pretrained; resume: tiếp tục epoch đã lưu; complete: đọc kết quả.
Lock theo ID/seed ngăn hai process ghi cùng một run. Chỉ chạy một notebook cho mỗi ID.
Khi OOM ở cặp, chạy model đó riêng với batch/config giữ nguyên; nếu chưa có last.pt, dùng EXP_ID mới.
Trên một GPU, chạy hai model chia sẻ compute/băng thông; thời gian mỗi epoch khi ghép cặp không dùng làm benchmark độc lập.

In [ ]:
status = nbtrain.training_status(cfg)
print("Trạng thái:", status)
if RUN_TRAINING and status != "complete":
    import subprocess
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,utilization.gpu", "--format=csv"],
        capture_output=True, text=True)
    print(gpu.stdout if gpu.returncode == 0 else
          "Không đọc được nvidia-smi. Đảm bảo kernel đang chạy trên host có CUDA.")
else:
    print("Đang xem kết quả; không cần khởi tạo CUDA ở kernel notebook.")

## 5. Train đủ epoch
Cell này stream log từng epoch từ process con. Checkpoint/log/curve/prediction có ID riêng.
Trong một cặp, bắt đầu cell này ở cả hai notebook có kernel độc lập.
Interrupt notebook sẽ dừng process con; lần chạy lại resume từ epoch hoàn tất gần nhất.
Không xoá last.pt hoặc thay scheduler/config để tiếp tục. Log nằm trong runs/notebook_logs/.

AMP có thể bỏ update và giảm scale khi gradient overflow; theo dõi loss và skip,
không dùng quy tắc tuyệt đối zero skip. Chạy lại code/amp_audit.py chỉ hỗ trợ các pilot;
mọi chẩn đoán thí nghiệm B cần dùng checkpoint/ID tương ứng.

In [ ]:
summary = None
if RUN_TRAINING:
    summary = nbtrain.run_in_subprocess(cfg, PLANNED_GROUP)
elif status == "complete":
    summary = nbtrain.verify_outputs(cfg)[0]
else:
    print("Chưa train đủ epoch. Bật RUN_TRAINING rồi chạy lại notebook trên host GPU.")
if summary is not None:
    display(summary)

## 6. Kiểm tra và lưu kết quả của model này
Xác nhận đủ 12 epoch, checkpoint chọn đúng, toàn bộ val và logits/CSV khớp nhau.
Kiểm tra việc test tắt. Ghi bảng riêng cho ID này để hai job không ghi đè nhau.
Metric val dùng cho sàng lọc backbone một seed; final/baseline ≥3 seed sẽ ở giai đoạn sau.

In [ ]:
history = None
if summary is not None:
    summary, history, metadata = nbtrain.verify_outputs(cfg)
    run_path = trainer.run_dir(cfg)
    row = {
        "exp_id":cfg.exp_id, "backbone":cfg.backbone,
        "pretrained_tag":metadata["pretrained_cfg"].get("tag"),
        "seed":cfg.seed, "epochs":cfg.epochs, "batch_size":cfg.batch_size,
        "img_size":cfg.img_size, "best_epoch":summary["best_epoch"],
        "val_macro_f1":summary["val"]["macro_f1"], "val_top1":summary["val"]["top1"],
        "val_ece":summary["val"]["ece"], "params_m":summary["params_m"], "gmacs":summary["gmacs"],
        "train_seconds_per_epoch_observed":summary["train_seconds_per_epoch"],
        "timing_scope":"user scheduling; may share GPU with another training job",
        "parallel_group_planned":PLANNED_GROUP,
        "total_skipped_updates":int(history["skipped_updates"].sum()),
        "completed_epochs":summary["completed_epochs"], "test_evaluated":False,
        "run_path":str(run_path.relative_to(ROOT)), "outputs_verified":True,
    }
    result_dir = CODE / "validation/backbones"
    result_dir.mkdir(parents=True, exist_ok=True)
    trainer.write_json(result_dir / f"{cfg.exp_id}_seed{cfg.seed}_checked.json", row)
    pd.DataFrame([row]).to_csv(result_dir / f"{cfg.exp_id}_seed{cfg.seed}_row.csv", index=False)
    display(pd.DataFrame([row]))
    display(history)
    print("Output audit: PASS")
else:
    print("Chưa có kết quả đủ epoch để xác minh hoặc ghi bảng backbone.")

## 7. Learning curves
Hình train đã lưu trong curves/. Hình dưới đây hiển thị loss và metric val qua các epoch.

In [ ]:
if history is not None:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(history["epoch"], history["train_loss"], label="Train CE")
    axes[0].plot(history["epoch"], history["val_loss"], label="Val CE")
    axes[1].plot(history["epoch"], history["val_macro_f1"], label="Val macro-F1")
    axes[1].plot(history["epoch"], history["val_top1"], label="Val top-1")
    axes[0].set(xlabel="Epoch",ylabel="CE")
    axes[1].set(xlabel="Epoch",ylabel="Score",ylim=(0,1))
    for ax in axes:
        ax.legend()
        ax.grid(alpha=.2)
    fig.suptitle(f"{cfg.exp_id} | {cfg.backbone} | T00 | seed {cfg.seed}")
    fig.tight_layout()
    plt.show()

## 8. Đọc tiến độ cả 5 model
Cell chỉ đọc các bảng đã xác minh, không ghi một file tổng dùng chung giữa các job.
Khi cả 5 hoàn tất, chọn backbone từ các run đủ epoch bằng macro-F1 val và cân nhắc params/GMAC.
Đo latency/throughput độc lập sau khi các job train đã kết thúc.

In [ ]:
progress = []
for exp_id,(backbone,group) in nbtrain.BACKBONES.items():
    path = CODE / "validation/backbones" / f"{exp_id}_seed{SEED}_checked.json"
    if path.is_file():
        saved = json.loads(path.read_text())
        progress.append({"exp_id":exp_id,"backbone":backbone,"group":group,
                         "status":"verified","val_macro_f1":saved["val_macro_f1"],
                         "val_top1":saved["val_top1"],"best_epoch":saved["best_epoch"]})
    else:
        progress.append({"exp_id":exp_id,"backbone":backbone,"group":group,"status":"pending"})
display(pd.DataFrame(progress))